In [1]:
import warnings
# 忽略所有 DeprecationWarning
warnings.filterwarnings("ignore", category=DeprecationWarning, module="scib|scgen|typing_extensions")

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import os
from scipy.sparse import issparse
import scvi
import scib
from scarches.models.scpoli import scPoli
import scarches as sca
from harmony import harmonize
import pyliger
from scarches.dataset.trvae.data_handling import remove_sparsity
import dask.dataframe as dd
import time
os.getcwd()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (
 captum (see https://github.com/pytorch/captum).
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/_settings.py:450: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  IPython.display.set_matplotlib_formats(*ipython_format)
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/umap/__init__.py:9: ImportWarning: T

'/home/lixiangyu/zr/Annotate/ANNOTATE_new/3_small_integration/3_small_integration_mouse_0518'

In this notebook the annotated datasets are concatinated, a ensembl <-> gene name mapping dictionary is created and different integration methods are implemented/executed on the dataset

# Mapping file for ensembl genes

# 1. Load and inspect the data

In [2]:
# indir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output/'
indir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/2_annotation/annot_mouse_0518/output_new_marker/'##用的新marker后的

In [15]:
# outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/3_small_integration/3_small_integration_mouse_0518/output_0518'
outdir = '/home/lixiangyu/zr/Annotate/ANNOTATE_new/3_small_integration/3_small_integration_mouse_0518/output_new_marker_0518'##用的新marker后的

#### Abdominal Aortic Aneurysm (AAA)

In [5]:
# data_GSE141031= sc.read_h5ad(os.path.join(indir,"GSE141031/GSE141031_annt_add.h5ad"))##新marker没有注释这个
data_GSE233625= sc.read_h5ad(os.path.join(indir,"GSE233625/GSE233625_annt_add.h5ad"))
data_GSE237067= sc.read_h5ad(os.path.join(indir,"GSE237067/GSE237067_annt_add.h5ad"))

#### Atherosclerosis

In [7]:
data_GSE269449= sc.read_h5ad(os.path.join(indir,"GSE269449/GSE269449_annt_add.h5ad"))
data_GSE284253= sc.read_h5ad(os.path.join(indir,"GSE284253/GSE284253_annt_add.h5ad"))
data_GSE279601= sc.read_h5ad(os.path.join(indir,"GSE279601/GSE279601_annt_add.h5ad"))

In [8]:
adata_list = [data_GSE233625,data_GSE237067,data_GSE269449,data_GSE284253,data_GSE279601]

In [9]:
adata_list

[AnnData object with n_obs × n_vars = 35479 × 32285
     obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'size_factors', 'leiden', 'cell_type_level1'
     var: 'original_ensembl_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
     uns: 'X_name', 'decontX', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'scDblFinder.class_colors', 'scDblFinder.threshold', 'umap'
     obsm: 'X_pca', 'X_umap', 'decontX_GSE233625_1_UMAP', 'decontX_GSE233625_2_UMAP', 'decontX_GSE233625_3_UMAP'
     varm: 'PCs'
     layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
     obsp: 'connectivities

In [10]:
adata_final = anndata.concat(adata_list, join='outer', fill_value=0.0)

In [11]:
adata_final.obs["cell_type_level1"].value_counts()

cell_type_level1
Smooth muscle cell       23335
Macrophage               13657
Fibroblast               13529
T cell                    8265
B cell                    6573
Dendritic cell            5271
Neutrophil                4059
Monocyte                  2578
Endothelial cell          2078
Natural killer cell        799
Erythrocyte/Erythroid       30
Name: count, dtype: int64

In [ ]:
# # rename
# adata_final.obs["cell_type_level1"] = adata_final.obs["cell_type_level1"].replace("B celll", "B cell")
# adata_final.obs["cell_type_level1"] = adata_final.obs["cell_type_level1"].replace("Natural killer cell ", "Natural killer cell")
# # adata_final.obs["cell_type_level1"] = adata_final.obs["cell_type_level1"].replace("Mase cell", "Mast cell")

In [12]:
#no neurons
adata_final.obs["cell_type_level1"].value_counts()

cell_type_level1
Smooth muscle cell       23335
Macrophage               13657
Fibroblast               13529
T cell                    8265
B cell                    6573
Dendritic cell            5271
Neutrophil                4059
Monocyte                  2578
Endothelial cell          2078
Natural killer cell        799
Erythrocyte/Erythroid       30
Name: count, dtype: int64

In [13]:
adata_final

AnnData object with n_obs × n_vars = 80174 × 34527
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'size_factors', 'leiden', 'cell_type_level1', 'n_counts'
    obsm: 'X_pca', 'X_umap', 'decontX_GSE233625_1_UMAP', 'decontX_GSE233625_2_UMAP', 'decontX_GSE233625_3_UMAP', 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP', 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP', 'decontX_GSE284253_1_UMAP', 'decontX_GSE284253_2_UMAP', 'decontX_GSE284253_3_UMAP', 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [16]:
# adata_final.write_h5ad(os.path.join(outdir,"mouse-small_concat.h5ad)"))
adata_final.write_h5ad(os.path.join(outdir,"mouse-small_concat-new_marker.h5ad)"))###new_marker

In [17]:
adata_final.var

""
0610005C13Rik
0610006L08Rik
0610007P14Rik
0610009B22Rik
0610009E02Rik
...
mt-Nd4
mt-Nd4l
mt-Nd5
mt-Nd6


# 2.1 Gene name mapping

In [18]:
# adata_final = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat.h5ad)"))
adata_final = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-new_marker.h5ad)"))###new_marker
adata_final

AnnData object with n_obs × n_vars = 80174 × 34527
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'size_factors', 'leiden', 'cell_type_level1', 'n_counts'
    obsm: 'X_pca', 'X_umap', 'decontX_GSE233625_1_UMAP', 'decontX_GSE233625_2_UMAP', 'decontX_GSE233625_3_UMAP', 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP', 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP', 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP', 'decontX_GSE284253_1_UMAP', 'decontX_GSE284253_2_UMAP', 'decontX_GSE284253_3_UMAP'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [19]:
ensembl_id_df = pd.read_csv("/home/lixiangyu/zr/Annotate/mouse_gene_names_to_ensembl.csv")

In [20]:
# Create a mapping from gene names to Ensembl IDs
gene_to_ensembl = dict(zip(ensembl_id_df['gene_name'], ensembl_id_df['ensembl_id']))

In [21]:
# Map the variable names in AnnData
adata_final.var['original_gene_names'] = adata_final.var_names
adata_final.var_names = [gene_to_ensembl[gene] if gene in gene_to_ensembl else gene for gene in adata_final.var_names]

In [22]:
non_ENSMUSG_vars = adata_final.var_names[~adata_final.var_names.str.startswith('ENSMUSG')]
len(non_ENSMUSG_vars)

2039

In [23]:
adata_final

AnnData object with n_obs × n_vars = 80174 × 34527
    obs: 'dataset', 'sample', 'symptoms', 'gender', 'age', 'intervention', 'location', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'size_factors', 'leiden', 'cell_type_level1', 'n_counts'
    var: 'original_gene_names'
    obsm: 'X_pca', 'X_umap', 'decontX_GSE233625_1_UMAP', 'decontX_GSE233625_2_UMAP', 'decontX_GSE233625_3_UMAP', 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP', 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP', 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP', 'decontX_GSE284253_1_UMAP', 'decontX_GSE284253_2_UMAP', 'decontX_GSE284253_3_UMAP'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [24]:
adata_final.var

,original_gene_names
ENSMUSG00000109644,0610005C13Rik
ENSMUSG00000108652,0610006L08Rik
0610007P14Rik,0610007P14Rik
0610009B22Rik,0610009B22Rik
ENSMUSG00000086714,0610009E02Rik
...,...
ENSMUSG00000064363,mt-Nd4
ENSMUSG00000065947,mt-Nd4l
ENSMUSG00000064367,mt-Nd5
ENSMUSG00000064368,mt-Nd6


In [25]:
#save .X in a new layer
adata_final.layers["log1p_scran_samplewise"] = adata_final.X.copy() 
# take the rounded_corrected_counts layer as .X
adata_final.X = adata_final.layers["rounded_corrected_counts"].copy()##把去污染后取整的计数作为x

In [26]:
# adata_final.write_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl.h5ad)"))
adata_final.write_h5ad(os.path.join(outdir,"mouse-small_concat-new_marker-ensembl.h5ad)"))

In [27]:
adata_final.var

,original_gene_names
ENSMUSG00000109644,0610005C13Rik
ENSMUSG00000108652,0610006L08Rik
0610007P14Rik,0610007P14Rik
0610009B22Rik,0610009B22Rik
ENSMUSG00000086714,0610009E02Rik
...,...
ENSMUSG00000064363,mt-Nd4
ENSMUSG00000065947,mt-Nd4l
ENSMUSG00000064367,mt-Nd5
ENSMUSG00000064368,mt-Nd6


In [28]:
# # Duplicate cells (before hvg)
# print(adata_final.layers["uncorrected_counts"].toarray().shape)
# print(np.unique(adata_final.layers["uncorrected_counts"].toarray(), axis=0).shape)
## no toarray
from scipy.sparse import csr_matrix
print(adata_final.layers["uncorrected_counts"].shape)
X = adata_final.layers["uncorrected_counts"]
X = X.tocsr()
row_signatures = set(
    zip(
        map(tuple, np.split(X.indices, X.indptr[1:-1])),
        map(tuple, np.split(X.data, X.indptr[1:-1]))
    )
)
print((len(row_signatures), X.shape[1]))


(80174, 34527)
(80174, 34527)


### plot dotplot
为了体现没有细胞在mast cell表达

In [30]:
adata_final.var['emsembl_id'] = adata_final.var_names
adata_final.var_names = adata_final.var['original_gene_names']
adata_final.var_names

Index(['0610005C13Rik', '0610006L08Rik', '0610007P14Rik', '0610009B22Rik',
       '0610009E02Rik', '0610009L18Rik', '0610009O20Rik', '0610010F05Rik',
       '0610010K14Rik', '0610011F06Rik',
       ...
       'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3', 'mt-Nd4', 'mt-Nd4l',
       'mt-Nd5', 'mt-Nd6', 'tdTomato'],
      dtype='object', name='original_gene_names', length=34527)

In [38]:
level1_marker = {
    'B cell': ['Cd79a', 'Cd79b', 'Ms4a1', 'Ly6d'],
    'Dendritic cell': ['Cd209a', 'H2-Ab1', 'H2-Eb1', 'Itgax', 'Flt3'],
    'Endothelial cell': ['Pecam1', 'Cdh5', 'Cldn5', 'Vwf'],
    'Erythrocyte/Erythroid': ['Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Alas2', 'Klf1'],
    'Fibroblast': ['Col1a1', 'Col1a2', 'Dcn', 'Lum'],
    'Macrophage': ['Cd68', 'Adgre1', 'C1qa', 'C1qb', 'Lyz2'],
    'Monocyte': ['Lyz2', 'Cd14', 'Ccr2', 'Ly6c2', 'Cx3cr1'],
    'Natural killer cell': ['Nkg7', 'Klrb1c', 'Gzma', 'Gzmb', 'Prf1'],
    'Neutrophil': ['S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Retnlg'],
    'Smooth muscle cell': ['Tagln', 'Myh11', 'Acta2', 'Cnn1', 'Myl9'],
    'T cell': ['Cd3d', 'Cd3g', 'Trbc2'],
    'Mast cell': ['Kit', 'Cpa3', 'Cma1', 'Mcpt4', 'Tpsb2'],
    'Basophil': ['Mcpt8', 'Cd200r3', 'Fcer1a', 'Prss34', 'Gata2'],
    'Pericyte': ['Rgs5', 'Pdgfrb', 'Wif1', 'Chad', 'Cspg4'],
}
#check if the marker genes are in the data and print percentage of marker genes per celltype in the data
#also print which marker genes are not in the data
#create new dict on the way
marker_genes_in_data = {}
for celltype in level1_marker:
    marker_genes_in_data[celltype] = list(set(level1_marker[celltype]).intersection(adata_final.var_names))
    print(celltype, len(set(level1_marker[celltype]).intersection(adata_final.var_names))/len(level1_marker[celltype]))
    print("not included: ", set(level1_marker[celltype]).difference(adata_final.var_names))

B cell 1.0
not included:  set()
Dendritic cell 1.0
not included:  set()
Endothelial cell 1.0
not included:  set()
Erythrocyte/Erythroid 1.0
not included:  set()
Fibroblast 1.0
not included:  set()
Macrophage 1.0
not included:  set()
Monocyte 1.0
not included:  set()
Natural killer cell 1.0
not included:  set()
Neutrophil 1.0
not included:  set()
Smooth muscle cell 1.0
not included:  set()
T cell 1.0
not included:  set()
Mast cell 1.0
not included:  set()
Basophil 1.0
not included:  set()
Pericyte 1.0
not included:  set()


In [39]:
dp = sc.pl.dotplot(
    adata_final,
    groupby="cell_type_level1",
    var_names=marker_genes_in_data,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    swap_axes = False,
    figsize = (30,3.5),
    return_fig = True
)

dp.legend(width=2.2)
dp.show()
dp.savefig("./dotplot_mouse.pdf")

# 2.2 Gene names aggregation

In [31]:
adata = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl.h5ad)")) 
adata

AnnData object with n_obs × n_vars = 58087 × 25939
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
    var: 'original_gene_names'
    obsm: 'X_pca', 'X_umap', 'decontX_GSE141031_apoe1_UMAP', 'decontX_GSE141031_apoe2_UMAP', 'decontX_GSE141031_apoe3_UMAP', 'decontX_GSE141031_apoe4_UMAP', 'decontX_GSE141031_dko1_UMAP', 'decontX_GSE141031_dko2_UMAP', 'decontX_GSE141031_dko3_UMAP', 'decontX_GSE141031_dko4_UMAP', 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP', 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP', 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP'

In [25]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
Smooth muscle cell       14125
Macrophage               10513
T cell                    8866
Monocyte                  7380
B cell                    6486
Fibroblast                3383
Dendritic cell            3263
Endothelial cell          1801
Natural killer cell        799
Neutrophil                 742
Pericyte                   699
Erythrocyte/Erythroid       30
Name: count, dtype: int64

In [32]:
adata.var

,original_gene_names
ENSMUSG00000109644,0610005C13Rik
ENSMUSG00000108652,0610006L08Rik
0610007P14Rik,0610007P14Rik
0610009B22Rik,0610009B22Rik
ENSMUSG00000086714,0610009E02Rik
...,...
ENSMUSG00000064368,mt-Nd6
ENSMUSG00000064337,mt-Rnr1
ENSMUSG00000064339,mt-Rnr2
ENSMUSG00000064372,mt-Tp


In [ ]:
#aggregation
# Convert the sparse matrix (if it is sparse) to a dense DataFrame
adata_df = pd.DataFrame(adata.X.toarray() if issparse(adata.X) else adata.X, 
                        index=adata.obs_names, 
                        columns=adata.var_names)

# Group by gene names and sum the counts
aggregated_data = adata_df.groupby(adata_df.columns, axis=1).sum()

# Prepare the new 'var' DataFrame, keeping the first occurrence of each gene
unique_var = adata.var.loc[~adata.var.index.duplicated(keep='first')]

# Create a new AnnData object with aggregated data
adata_agg = anndata.AnnData(X=aggregated_data, obs=adata.obs, var=unique_var.loc[aggregated_data.columns])

# 'adata_agg' now has unique gene names and aggregated counts

In [28]:
adata_agg

AnnData object with n_obs × n_vars = 58087 × 25939
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
    var: 'original_gene_names'

In [30]:
adata_agg.var

,original_gene_names
0610007P14Rik,0610007P14Rik
0610009B22Rik,0610009B22Rik
0610009O20Rik,0610009O20Rik
0610010F05Rik,0610010F05Rik
0610011F06Rik,0610011F06Rik
...,...
Zufsp,Zufsp
ccdc198,ccdc198
eGFP,eGFP
l7Rn6,l7Rn6


In [6]:
# MANUALLY CHECK IF SUMMATION WORKED AS INTENDED(做验证)
# Extract counts for "ENSMUSG00000086714" from the original adata
original_counts = adata[:, "ENSMUSG00000086714"].X
if issparse(original_counts):
    original_counts = original_counts.toarray()  # Convert to dense array if sparse

# Sum these counts cellwise
summed_counts = np.sum(original_counts, axis=1)

# Extract counts for "ENSMUSG00000086714" from adata_agg
agg_counts = adata_agg[:, "ENSMUSG00000086714"].X
if issparse(agg_counts):
    agg_counts = agg_counts.toarray()  # Convert to dense array if sparse

# Compare the two sets of counts
comparison = np.allclose(summed_counts.flatten(), agg_counts.flatten())

# Print the result of the comparison
print(f"The counts are correctly summed: {comparison}")


The counts are correctly summed: True


In [ ]:
# Function to aggregate a layer
def aggregate_layer(layer):
    layer_df = pd.DataFrame(layer.toarray() if issparse(layer) else layer, 
                            index=adata.obs_names, 
                            columns=adata.var_names)
    return layer_df.groupby(layer_df.columns, axis=1).sum()

# Aggregate each layer and add to adata_agg
for layer_name in adata.layers.keys():
    print(f"Aggregating layer: {layer_name}")
    aggregated_layer = aggregate_layer(adata.layers[layer_name])
    
    # The first time, we need to initialize layers in adata_agg
    if not hasattr(adata_agg, 'layers'):
        adata_agg.layers = {}

    # Add the aggregated layer to adata_agg
    adata_agg.layers[layer_name] = aggregated_layer

# Now 'adata_agg' contains all the aggregated layers
####.py

In [5]:
adata = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl.h5ad")) 
adata_agg = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl-agg.h5ad")) 

In [7]:
adata_agg

AnnData object with n_obs × n_vars = 58087 × 25939
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
    var: 'original_gene_names'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [8]:
adata

AnnData object with n_obs × n_vars = 58087 × 25939
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
    var: 'original_gene_names'
    obsm: 'X_pca', 'X_umap', 'decontX_GSE141031_apoe1_UMAP', 'decontX_GSE141031_apoe2_UMAP', 'decontX_GSE141031_apoe3_UMAP', 'decontX_GSE141031_apoe4_UMAP', 'decontX_GSE141031_dko1_UMAP', 'decontX_GSE141031_dko2_UMAP', 'decontX_GSE141031_dko3_UMAP', 'decontX_GSE141031_dko4_UMAP', 'decontX_GSE237067_1_UMAP', 'decontX_GSE237067_2_UMAP', 'decontX_GSE237067_3_UMAP', 'decontX_GSE269449_1_UMAP', 'decontX_GSE269449_2_UMAP', 'decontX_GSE279601_1_UMAP', 'decontX_GSE279601_2_UMAP', 'decontX_GSE279601_3_UMAP', 'decontX_GSE279601_4_UMAP'

In [9]:
def sum_gene_counts(layer, gene_name, var_names):
    # Find the index(es) of the gene
    gene_indices = np.where(var_names == gene_name)[0]

    # Sum counts across all occurrences of the gene
    gene_counts = np.sum(layer[:, gene_indices].toarray(), axis=1) if issparse(layer) else np.sum(layer[:, gene_indices], axis=1)
    return gene_counts

# Check for each layer
for layer_name in adata.layers.keys():
    print(f"Checking layer: {layer_name}")

    # Get var names for the current layer
    var_names = adata.var_names

    # Sum counts for "ENSMUSG00000086714" in the original data
    original_counts = sum_gene_counts(adata.layers[layer_name], "ENSMUSG00000086714", var_names)

    # Extract counts for "ENSMUSG00000086714" from adata_agg layer
    gene_index_agg = np.where(adata_agg.var_names == "ENSMUSG00000086714")[0]
    agg_counts = adata_agg.layers[layer_name][:, gene_index_agg]
    if issparse(agg_counts):
        agg_counts = agg_counts.toarray()

    # Compare the two sets of counts
    comparison = np.allclose(original_counts.flatten(), agg_counts.flatten())

    # Print the result of the comparison
    print(f"The counts are correctly summed in layer {layer_name}: {comparison}")

Checking layer: log1p_scran_samplewise
The counts are correctly summed in layer log1p_scran_samplewise: True
Checking layer: raw_decontXcounts
The counts are correctly summed in layer raw_decontXcounts: True
Checking layer: rounded_corrected_counts
The counts are correctly summed in layer rounded_corrected_counts: True
Checking layer: uncorrected_counts
The counts are correctly summed in layer uncorrected_counts: True


In [10]:
adata_agg

AnnData object with n_obs × n_vars = 58087 × 25939
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
    var: 'original_gene_names'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [11]:
import scipy.sparse as sparse

In [12]:
# 转成稀疏
adata_agg.X = sparse.csr_matrix(adata_agg.X)
for layer_name in adata_agg.layers.keys():
    print("converting to sparse matrix in", layer_name)
    adata_agg.layers[layer_name] = sparse.csr_matrix(adata_agg.layers[layer_name])


converting to sparse matrix in log1p_scran_samplewise
converting to sparse matrix in raw_decontXcounts
converting to sparse matrix in rounded_corrected_counts
converting to sparse matrix in uncorrected_counts


In [13]:
adata_agg.write(os.path.join(outdir,"mouse-small_concat-ensembl-agg-sparse.h5ad")) 


# 3. Normalization

In [14]:
adata_final = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl-agg-sparse.h5ad")) 

In [15]:
adata_final

AnnData object with n_obs × n_vars = 58087 × 25939
    obs: 'sample', 'dataset', 'symptoms', 'gender', 'location', 'age', 'intervention', 'scDblFinder.sample', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1'
    var: 'original_gene_names'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [16]:
#Perform a clustering for scran normalization in clusters
adata_pp = adata_final.copy()
sc.pp.normalize_total(adata_pp, target_sum=1e6)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp, svd_solver="arpack")
sc.pp.neighbors(adata_pp, n_pcs=30)
sc.tl.leiden(adata_pp, key_added='groups', resolution=0.22)

In [17]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

# Ignore R warning messages
#Note: this can be commented out to get more verbose R output
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Automatically convert rpy2 outputs to pandas dataframes
pandas2ri.activate()
anndata2ri.activate()
%load_ext rpy2.ipython

In [18]:
#Preprocess variables for scran normalization
input_groups = adata_pp.obs['groups']
data_mat = adata_final.X.T.toarray()
# data_mat = adata_final.X.T 

In [19]:
%%R -i data_mat -i input_groups -o size_factors
library(scran)
size_factors = calculateSumFactors(data_mat, clusters=input_groups, min.mean=0.1)

Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOr

In [20]:
del adata_pp
adata_final.obs['size_factors'] = size_factors

In [21]:
#Normalize adata 
adata_final.X /= adata_final.obs['size_factors'].values[:,None]
sc.pp.log1p(adata_final)

In [22]:
adata_agg.write(os.path.join(outdir,"mouse-small_concat-ensembl-agg-sparse-scranlog1p.h5ad")) 

In [ ]:
adata_final = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl-agg-sparse-scranlog1p.h5ad")) 
adata_final

AnnData object with n_obs × n_vars = 475511 × 35426
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue'
    var: 'original_gene_names'
    uns: 'log1p'
    layers: 'log1p_scran_samplewise', 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'

In [5]:
adata_final.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   156616
Neutrophil                69784
Macrophage                48974
Fibroblast                39743
Endothelial cell          35131
B cell                    34850
Smooth muscle cell        31767
Natural killer cell       20082
Monocyte                  16491
Dendritic cell             9888
Pericyte                   9219
Mast cell                  2538
Erythrocyte/Erythroid       309
Basophil                    119
Name: count, dtype: int64

# 4. Integrate

In [ ]:
adata_final = sc.read_h5ad(os.path.join(outdir,"mouse-small_concat-ensembl-agg-sparse-scranlog1p.h5ad")) 

In [4]:
def prepare_adata(adata_final):
    
    #delete all layers that are not needed to reduce memory
    del adata_final.layers['log1p_scran_samplewise'] #only for small integrations
    del adata_final.layers['raw_decontXcounts']
    del adata_final.layers['uncorrected_counts']

    #rename layer to counts
    adata_final.layers['counts'] = adata_final.layers['rounded_corrected_counts']
    del adata_final.layers['rounded_corrected_counts']

    # add "_{dataset}" to the sample name to make them unique
    adata_final.obs["sample"] = adata_final.obs["sample"].astype(str) + "_" + adata_final.obs["dataset"].astype(str)
    adata_final.obs['sample'] = adata_final.obs['sample'].astype('category')   

In [5]:
def preprocess_adata(adata_final, mode, batch_key):

    # remove unknown and doublets
    print("Removing unknowns and doublets...")
    adata_final = adata_final[~adata_final.obs['cell_type_level1'].isin(['unknown', 'doublets'])].copy()

    
    if mode=="auto":
        print("Using auto mode to do hvg and pca...")
        scib.preprocessing.reduce_data(adata_final, batch_key=batch_key)
        
    elif mode=="manual":
        print("Using manual mode to do hvg and pca...")
        sc.pp.highly_variable_genes(adata_final,  n_top_genes = 2000, batch_key = batch_key)
        sc.pp.pca(adata_final, use_highly_variable=True)

    else:
        raise ValueError("Mode must be 'auto' or 'manual'")

    
    # subset only hvgs
    print("Subset for HVGs...")
    hvg = adata_final.var[adata_final.var['highly_variable']].index.tolist()
    adata_final = adata_final[:, hvg].copy()

    
    #check how many cells have zero counts for all genes
    print("Check how many cells have zero counts for all genes...")
    cellwise_sum = adata_final.X.sum(axis=1)
    num_cells_zero_counts = (cellwise_sum == 0).sum()
    
    if num_cells_zero_counts>0:
        print(num_cells_zero_counts, " cells were found with 0 counts across all genes! Removing these cells now...")
        adata_final = adata_final[cellwise_sum > 0, :]

    
    # check for dublicate gene expressions across cells after HVG selection
    print("Check for dublicate gene expressions")
    before = adata_final.layers["counts"].toarray().shape[0]
    after = np.unique(adata_final.layers["counts"].toarray(), axis=0).shape[0]
    diff = before - after

    if diff > 0:
        print(diff, " Non unique cell expression profiles found! Removing them...")
        
        counts_array = adata_final.layers["counts"].toarray()
        # Find the indices of unique rows
        _, unique_indices = np.unique(counts_array, axis=0, return_index=True)
        # Sort the indices to maintain the original order
        unique_indices_sorted = np.sort(unique_indices)
        # Filter the AnnData object to keep only unique rows
        adata_final = adata_final[unique_indices_sorted]
    else:
        "No non unique cells found."

    print("Preprocessing finished!")  

    return adata_final

In [6]:
def integrate_methods(adata, batch_key, cell_type, scgen_impl):
    
    # adata has to be filtered, hvg filtered and pca applied in obsm[X_pca]
    # log normalized data in .X and counts in .layers["counts"] 

    
    adata_final = adata

    # scGen
    print("Integrating with scGen...")

    if scgen_impl == "scib":
        print("scGen scib implementation")
        adata_after = scib.ig.scgen(adata_final, batch=batch_key, cell_type=cell_type)
        adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
        
    elif scgen_impl == "scarches":
        print("scGen scarches implementation")

        adata_final = remove_sparsity(adata_final) # remove sparsity
        adata_final.X = adata_final.X.astype("float32")
        
        epoch = 100
        early_stopping_kwargs = {
            "early_stopping_metric": "val_loss",
            "patience": 25,
            "threshold": 0,
            "reduce_lr": True,
            "lr_patience": 13,
            "lr_factor": 0.1,
        }

        network = sca.models.scgen(adata = adata_final,
                           hidden_layer_sizes=[256,128])

        network.train(n_epochs=epoch, early_stopping_kwargs = early_stopping_kwargs, batch_size = 32)

        adata_after = network.batch_removal(adata_final, batch_key=batch_key, cell_label_key=cell_type,return_latent=True)
        adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
    
    
    # scVI
    print("Integrating with scVI...")
    
    scvi.model.SCVI.setup_anndata(adata_final, layer="counts", batch_key=batch_key)
    vae = scvi.model.SCVI(adata_final, gene_likelihood="nb", n_layers=2, n_latent=30)
    vae.train()
    adata_final.obsm["scVI"] = vae.get_latent_representation()

    
    # scANVI
    print("Integrating with scANVI...")
    
    lvae = scvi.model.SCANVI.from_scvi_model(
        vae,
        #adata=adata_final,
        labels_key=cell_type,
        unlabeled_category="none"
    )
    lvae.train(max_epochs=40)
    adata_final.obsm["scANVI"] = lvae.get_latent_representation()

    
    # scPoli
    print("Integrating with scPoli...")
    early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
    }
    
    scpoli_model = scPoli(
    adata=adata_final,
    condition_keys=batch_key,
    cell_type_keys=cell_type,
    embedding_dims=5,
    recon_loss='nb',
    )
    
    scpoli_model.train(
        n_epochs=50,
        pretraining_epochs=40,
        early_stopping_kwargs=early_stopping_kwargs,
        eta=5,
    )

    scpoli_latent = scpoli_model.get_latent(adata_final)
    adata_final.obsm["scPoli"] = scpoli_latent

    
    # Harmony
    print("Integrating with Harmony...")
    adata_final.obsm["Harmony"] = harmonize(adata_final.obsm["X_pca"], adata_final.obs, batch_key=batch_key)

    
    # LIGER
    print("Integrating with LIGER...")

    batch_cats = adata_final.obs["sample"].cat.categories
    bdata = adata_final.copy()
    # Pyliger normalizes by library size with a size factor of 1
    # So here we give it the count data
    bdata.X = bdata.layers["counts"]
    # List of adata per dataset
    adata_list = [bdata[bdata.obs[batch_key] == b].copy() for b in batch_cats]
    for i, ad in enumerate(adata_list):
        ad.uns["sample_name"] = batch_cats[i]
        # Hack to make sure each method uses the same genes
        ad.uns["var_gene_idx"] = np.arange(bdata.n_vars)
    
    
    liger_data = pyliger.create_liger(adata_list, remove_missing=False, make_sparse=False)
    # Hack to make sure each method uses the same genes
    liger_data.var_genes = bdata.var_names
    pyliger.normalize(liger_data)
    pyliger.scale_not_center(liger_data)
    pyliger.optimize_ALS(liger_data, k=30)
    pyliger.quantile_norm(liger_data)
    
    
    adata_final.obsm["LIGER"] = np.zeros((adata_final.shape[0], liger_data.adata_list[0].obsm["H_norm"].shape[1]))
    for i, b in enumerate(batch_cats):
        adata_final.obsm["LIGER"][adata_final.obs[batch_key] == b] = liger_data.adata_list[i].obsm["H_norm"]

    print("Finished integrating the data")
    
    return adata_final

In [7]:
def integrate_sca(adata, batch_key, cell_type,):
    
    # adata has to be filtered, hvg filtered and pca applied in obsm[X_pca]
    # log normalized data in .X and counts in .layers["counts"] 

    
    adata_final = adata

    adata_counts = adata_final.copy()
    adata_counts.X = adata_counts.layers["counts"].copy()

    # print("scGen scib implementation")
    # adata_after = scib.ig.scgen(adata_final, batch=batch_key, cell_type=cell_type)
    # adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
    
    '''
    # scGen too slow
    print("Integrating with scGen...")

    adata_final = remove_sparsity(adata_final) # remove sparsity
    adata_final.X = adata_final.X.astype("float32")
    
    epoch = 100
    early_stopping_kwargs = {
        "early_stopping_metric": "val_loss",
        "patience": 25,
        "threshold": 0,
        "reduce_lr": True,
        "lr_patience": 13,
        "lr_factor": 0.1,
    }

    network = sca.models.scgen(adata = adata_final,
                       hidden_layer_sizes=[256,128])

    network.train(n_epochs=epoch, early_stopping_kwargs = early_stopping_kwargs, batch_size = 32)

    adata_after = network.batch_removal(adata_final, batch_key=batch_key, cell_label_key=cell_type,return_latent=True)
    adata_final.obsm["scGen"] = adata_after.obsm["corrected_latent"].copy()
    '''

    
    # scVI
    print("Integrating with scVI...")

    sca.models.SCVI.setup_anndata(adata_counts, batch_key=batch_key)

    vae = sca.models.SCVI(
        adata_counts,
        n_layers=2,##默认为1
        encode_covariates=True,
        deeply_inject_covariates=False,
        use_layer_norm="both",
        use_batch_norm="none",
    )

    vae.train()
    adata_final.obsm["scVI"] = vae.get_latent_representation()

    
    # scANVI
    print("Integrating with scANVI...")

    scanvae = sca.models.SCANVI.from_scvi_model(vae, unlabeled_category = "none", labels_key=cell_type)
    scanvae.train(max_epochs=40)
    adata_final.obsm["scANVI"] = scanvae.get_latent_representation()##采用训练好的scvi模型

    
    
    # scPoli
    print("Integrating with scPoli...")
    early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
    }
    
    scpoli_model = scPoli(
    adata=adata_final,
    condition_keys=batch_key,
    cell_type_keys=cell_type,
    embedding_dims=5,
    recon_loss='mse',
    )
    
    scpoli_model.train(
        n_epochs=50,##默认为100
        pretraining_epochs=40,
        early_stopping_kwargs=early_stopping_kwargs,
        eta=5,##默认为1
    )

    scpoli_latent = scpoli_model.get_latent(adata_final)
    adata_final.obsm["scPoli"] = scpoli_latent


    '''
    ####################################################################################
    # scPoli counts
    print("Integrating with scPoli using raw counts...")
    early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
    }
    
    scpoli_model2 = scPoli(
    adata=adata_counts,
    condition_keys=batch_key,
    cell_type_keys=cell_type,
    embedding_dims=5,
    recon_loss='nb',
    )
    
    scpoli_model2.train(
        n_epochs=50,
        pretraining_epochs=40,
        early_stopping_kwargs=early_stopping_kwargs,
        eta=5,
    )

    scpoli_latent2 = scpoli_model2.get_latent(adata_counts)
    adata_final.obsm["scPoli_counts"] = scpoli_latent2
    ####################################################################################
    '''
    
    # Harmony
    print("Integrating with Harmony...")
    adata_final.obsm["Harmony"] = harmonize(adata_final.obsm["X_pca"], adata_final.obs, batch_key=batch_key)

    
    # LIGER
    print("Integrating with LIGER...")

    batch_cats = adata_final.obs["sample"].cat.categories
    bdata = adata_final.copy()
    # Pyliger normalizes by library size with a size factor of 1
    # So here we give it the count data
    bdata.X = bdata.layers["counts"]
    # List of adata per dataset
    adata_list = [bdata[bdata.obs[batch_key] == b].copy() for b in batch_cats]
    for i, ad in enumerate(adata_list):
        ad.uns["sample_name"] = batch_cats[i]
        # Hack to make sure each method uses the same genes
        ad.uns["var_gene_idx"] = np.arange(bdata.n_vars)
    
    
    liger_data = pyliger.create_liger(adata_list, remove_missing=False, make_sparse=False)
    # Hack to make sure each method uses the same genes
    liger_data.var_genes = bdata.var_names
    pyliger.normalize(liger_data)
    pyliger.scale_not_center(liger_data)
    pyliger.optimize_ALS(liger_data, k=30)
    pyliger.quantile_norm(liger_data)
    
    
    adata_final.obsm["LIGER"] = np.zeros((adata_final.shape[0], liger_data.adata_list[0].obsm["H_norm"].shape[1]))
    for i, b in enumerate(batch_cats):
        adata_final.obsm["LIGER"][adata_final.obs[batch_key] == b] = liger_data.adata_list[i].obsm["H_norm"]


    
    print("Finished integrating the data")
    
    return adata_final

In [8]:
import time
start_time = time.time()

In [9]:
prepare_adata(adata_final)

In [ ]:
adata_final

In [ ]:
adata_final = preprocess_adata(adata_final, mode = "auto", batch_key="sample")

In [ ]:
adata_final = integrate_sca(adata_final, batch_key = "sample", cell_type = "cell_type_level1")##no-scgen(版本问题)

In [ ]:
end_time = time.time()
print(f"Elapsed time: {end_time - start_time} seconds")

In [40]:
# check for dublicates in embeddings

In [ ]:
for embed in ['Harmony', 'LIGER', 'X_pca', 'scANVI', 'scPoli', 'scVI']:
    diff = adata_final.obsm[embed].shape[0] - np.unique(adata_final.obsm[embed], axis=0).shape[0]
    print(diff," for ", embed )

In [13]:
#rename obsm X_pca to PCA in adata_final
adata_final.obsm["PCA"] = adata_final.obsm["X_pca"]
del adata_final.obsm["X_pca"]

In [14]:
# adata_final.write_h5ad("./output_1226/small-concat-aggr-sparse-method-mse.h5ad") # used in the 3-1_evaluation-HPC.py script
adata_final.write_h5ad("./output_1226/small-concat-aggr-sparse-method-mse-noCxG.h5ad") # used in the 3-1_evaluation-HPC.py script

# UMAP-dim=15 eta=10 mse

In [3]:
adata = sc.read_h5ad('/home/lixiangyu/zr/Annotate/ANNOTATE_new/3_small_integration/3_3_train_py_no_IAISR_0507/output_260420/small-concat-ensembl-aggr-sparse-method-noIAISR-mse-dim15-eta10.h5ad')

In [4]:
adata

AnnData object with n_obs × n_vars = 475147 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'conditions_combined'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'Harmony', 'LIGER', 'PCA', 'scANVI', 'scPoli', 'scVI'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [5]:
adata.obs

,sample,dataset,symptoms,scDblFinder.class,scDblFinder.score,scDblFinder.weighted,scDblFinder.cxds_score,decontX_contamination,decontX_clusters,n_genes_by_counts,...,n_counts,size_factors,leiden,cell_type_level1,scDblFinder.sample,gender,age,intervention,tissue,conditions_combined
AAACCCAAGAGGTTAT-1,1_JD_1_JD,1_JD,not stated,singlet,0.003171,0.399183,0.006773,0.002891,1,2300,...,8822.0,1.843081,13,Dendritic cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD
AAACCCAAGCAACAAT-1,1_JD_1_JD,1_JD,not stated,singlet,0.001880,0.346668,0.013342,0.002804,1,2848,...,10207.0,2.564145,10,Dendritic cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD
AAACCCAAGGAGTCTG-1,1_JD_1_JD,1_JD,not stated,singlet,0.000222,0.091969,0.015069,0.029719,2,963,...,2989.0,0.556303,0,B cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD
AAACCCACAACTCATG-1,1_JD_1_JD,1_JD,not stated,singlet,0.035987,0.272248,0.105747,0.024003,1,2267,...,9232.0,1.904665,13,Dendritic cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD
AAACCCACAGCTCTGG-1,1_JD_1_JD,1_JD,not stated,singlet,0.000791,0.208679,0.001585,0.007463,3,1502,...,4100.0,0.981015,19,Fibroblast,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Hu_TTTGTCAGTGAAAGAG-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,singlet,0.000004,0.057483,0.006366,0.007540,S3-12,296,...,1351.0,0.003169,19,Natural killer cell,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.
Hu_TTTGTCAGTGATGCCC-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,singlet,0.000009,0.071639,0.002198,0.000256,S3-3,227,...,732.0,0.014026,2,Fibroblast,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.
Hu_TTTGTCATCAGTGTTG-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,doublet,0.968598,0.429082,0.096722,0.013633,S3-2,1077,...,6410.0,2.508845,13,Macrophage,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.
Hu_TTTGTCATCGCGCCAA-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,doublet,0.991784,0.234881,0.166324,0.149965,S3-2,609,...,3474.0,0.939092,23,Dendritic cell,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.


In [ ]:
# adata.obs_names = adata.obs_names.str.replace(r'(-\d+)+$', '', regex=True)
# adata.obs

,sample,dataset,symptoms,scDblFinder.class,scDblFinder.score,scDblFinder.weighted,scDblFinder.cxds_score,decontX_contamination,decontX_clusters,n_genes_by_counts,...,leiden,cell_type_level1,scDblFinder.sample,gender,age,intervention,tissue,conditions_combined,clean_cell_id,ground_truth
AAACCCAAGAGGTTAT,1_JD_1_JD,1_JD,not stated,singlet,0.003171,0.399183,0.006773,0.002891,1,2300,...,13,Dendritic cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD,AAACCCAAGAGGTTAT-1,NaN
AAACCCAAGCAACAAT,1_JD_1_JD,1_JD,not stated,singlet,0.001880,0.346668,0.013342,0.002804,1,2848,...,10,Dendritic cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD,AAACCCAAGCAACAAT-1,NaN
AAACCCAAGGAGTCTG,1_JD_1_JD,1_JD,not stated,singlet,0.000222,0.091969,0.015069,0.029719,2,963,...,0,B cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD,AAACCCAAGGAGTCTG-1,NaN
AAACCCACAACTCATG,1_JD_1_JD,1_JD,not stated,singlet,0.035987,0.272248,0.105747,0.024003,1,2267,...,13,Dendritic cell,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD,AAACCCACAACTCATG-1,NaN
AAACCCACAGCTCTGG,1_JD_1_JD,1_JD,not stated,singlet,0.000791,0.208679,0.001585,0.007463,3,1502,...,19,Fibroblast,NaN,NaN,NaN,NaN,NaN,1_JD_1_JD,AAACCCACAGCTCTGG-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Hu_TTTGTCAGTGAAAGAG-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,singlet,0.000004,0.057483,0.006366,0.007540,S3-12,296,...,19,Natural killer cell,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.,Hu_TTTGTCAGTGAAAGAG-S3_CA2,NaN
Hu_TTTGTCAGTGATGCCC-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,singlet,0.000009,0.071639,0.002198,0.000256,S3-3,227,...,2,Fibroblast,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.,Hu_TTTGTCAGTGATGCCC-S3_CA2,NaN
Hu_TTTGTCATCAGTGTTG-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,doublet,0.968598,0.429082,0.096722,0.013633,S3-2,1077,...,13,Macrophage,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.,Hu_TTTGTCATCAGTGTTG-S3_CA2,NaN
Hu_TTTGTCATCGCGCCAA-S3_CA2,S3_Bleckwehl T et al.,Bleckwehl T et al.,Healthy,doublet,0.991784,0.234881,0.166324,0.149965,S3-2,609,...,23,Dendritic cell,S3,Female,Unknown,Unknown,Coronary artery,S3_Bleckwehl T et al.,Hu_TTTGTCATCGCGCCAA-S3_CA2,NaN


## harmony

In [ ]:
# import pandas as pd
# import scanpy as sc

# csv_data = pd.read_csv(
#     "/home/lixiangyu/zr/Annotate/ground_truth_manual.csv",
#     header=0
# )

# # 清理 ground truth 里的细胞 ID
# csv_data.iloc[:, 0] = csv_data.iloc[:, 0].astype(str).str.replace(
#     r'(-\d+[_]\d+[_]\d+)+$',
#     '',
#     regex=True
# )

# # 统一命名
# csv_data.iloc[:, 1] = csv_data.iloc[:, 1].replace({
#     "SMC": "SMC(Fibromyocyte)",
#     "Fibromyocyte": "SMC(Fibromyocyte)",
#     "B_cell": "B_cell(Plasma_cell)",
#     "Plasma_cell": "B_cell(Plasma_cell)",
# })

# # 去重并设置 index
# csv_data = csv_data.iloc[:, :2].copy()
# csv_data.columns = ["cell_id", "ground_truth"]
# csv_data.drop_duplicates(subset="cell_id", keep="first", inplace=True)
# csv_data = csv_data.set_index("cell_id")

# print("\nGround truth counts:")
# print(csv_data["ground_truth"].value_counts())

# # 同样清理 adata.obs_names，生成一个用于匹配的 clean_cell_id
# adata.obs["clean_cell_id"] = adata.obs_names.astype(str).str.replace(
#     r'(-\d+[_]\d+[_]\d+)+$',
#     '',
#     regex=True
# )

# # 把 ground truth 映射到 adata.obs
# adata.obs["ground_truth"] = adata.obs["clean_cell_id"].map(csv_data["ground_truth"])

# print("\nMatched ground truth counts in adata:")
# print(adata.obs["ground_truth"].value_counts(dropna=False))

# # 用 scPoli embedding 画 UMAP
# adata.obsm["X_pca"] = adata.obsm["Harmony"]
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)

# # 画 ground truth，不用 cell_type_level1
# sc.pl.umap(
#     adata,
#     color="ground_truth",
#     title="scPoli_ground_truth",
#     save="_ground_truth_dim10_eta5_Harmony.png"
# )

# sc.pl.umap(
#     adata,
#     color="sample",
#     title="scPoli_sample",
#     save="_sample_dim10_eta5_Harmony.png"
# )

# # IAISR 子集
# adata_IAISR = adata[adata.obs["dataset"] == "IAISR"].copy()

# sc.pl.umap(
#     adata_IAISR,
#     color="ground_truth",
#     title="scPoli_IAISR_ground_truth",
#     save="_total_IAISR_ground_truth_dim10_eta5_Harmony.png"
# )

# sc.pl.umap(
#     adata_IAISR,
#     color="sample",
#     title="scPoli_IAISR_sample",
#     save="_total_IAISR_sample_dim10_eta5_Harmony.png"
# )


Ground truth counts:
ground_truth
T_cell                 83305
Monocyte               21864
SMC(Fibromyocyte)      21688
B_cell(Plasma_cell)    21360
Endothelial_cell       19305
Neutrophil             18708
Macrophage             18026
NK_cell                 8398
DC                      7449
Fibroblast              5757
Mast_cell               2044
NKT_cell                1437
Name: count, dtype: int64

Matched ground truth counts in adata:
ground_truth
NaN                    259900
T_cell                  86112
Monocyte                22898
SMC(Fibromyocyte)       21618
B_cell(Plasma_cell)     21589
Neutrophil              19600
Endothelial_cell        19524
Macrophage              18511
NK_cell                  9095
DC                       7431
Fibroblast               5254
Mast_cell                2137
NKT_cell                 1475
Name: count, dtype: int64


In [7]:
adata.obsm['X_pca'] = adata.obsm['Harmony']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="harmony_cell_type1",save='_cell_type_dim15_eta10-harmony.png')
sc.pl.umap(adata, color='sample',title="harmony_sample",save='_sample_dim15_eta10-harmony.png')


# adata_IAISR = adata[adata.obs['dataset'] == 'IAISR'].copy()

# sc.pl.umap(
#     adata_IAISR,
#     color='cell_type_level1',
#     title="harmony_IAISR_cell_type1",
#     save='_total_IAISR_cell_type_dim15_eta10_harmony.png'
# )

# sc.pl.umap(
#     adata_IAISR,
#     color='sample',
#     title="harmony_IAISR_sample",
#     save='_total_IAISR_sample_dim15_eta10_harmony.png')

In [7]:
import scanpy as sc

# 筛选 IAISR 细胞
adata_iaisr = adata[
    adata.obs["dataset"] == "IAISR"
].copy()

print(adata_iaisr)
print(adata_iaisr.obs["cell_type_level1"].value_counts())

# 使用 Harmony embedding
adata_iaisr.obsm["X_pca"] = adata_iaisr.obsm["Harmony"]

# 重新计算 neighbors 和 UMAP
sc.pp.neighbors(
    adata_iaisr,
    use_rep="Harmony"
)
sc.tl.umap(adata_iaisr)
sc.pl.umap(
    adata_iaisr,
    color="sample",
    title="IAISR - Harmony by sample",
    save="_IAISR_sample_harmony.png"
)
# 如果还有更细的 annotation，也可以一起看
sc.pl.umap(
    adata_iaisr,
    color="cell_type_level1",
    title="IAISR - Harmony cell_type_level1",
    save="_IAISR_cell_type_level1_harmony.png"
)

AnnData object with n_obs × n_vars = 19996 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'conditions_combined'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'Harmony', 'LIGER', 'PCA', 'scANVI', 'scPoli', 'scVI'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
cell_type_level1
Neutrophil             12418
Dendritic cell          2724
Fibroblast              1478
T cell                  1473
Natural killer cell     1115
B cell                   788
Name: count, dtype: int64


## scANVI

In [8]:
adata.obsm['X_pca'] = adata.obsm['scANVI']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scANVI_cell_type1",save='_cell_type_dim15_eta10-scANVI.png')
sc.pl.umap(adata, color='sample',title="scANVI_sample",save='_sample_dim15_eta10-scANVI.png')

In [6]:
import scanpy as sc

# 筛选 IAISR 细胞
adata_iaisr = adata[
    adata.obs["dataset"] == "IAISR"
].copy()

print(adata_iaisr)
print(adata_iaisr.obs["cell_type_level1"].value_counts())

# 使用 scANVI embedding
adata_iaisr.obsm["X_pca"] = adata_iaisr.obsm["scANVI"]

# 重新计算 neighbors 和 UMAP
sc.pp.neighbors(
    adata_iaisr,
    use_rep="scANVI"
)
sc.tl.umap(adata_iaisr)
sc.pl.umap(
    adata_iaisr,
    color="sample",
    title="IAISR - scANVI by sample",
    save="_IAISR_sample_scANVI.png"
)
# 如果还有更细的 annotation，也可以一起看
sc.pl.umap(
    adata_iaisr,
    color="cell_type_level1",
    title="IAISR - scANVI cell_type_level1",
    save="_IAISR_cell_type_level1_scANVI.png"
)

AnnData object with n_obs × n_vars = 19996 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'conditions_combined'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'Harmony', 'LIGER', 'PCA', 'scANVI', 'scPoli', 'scVI'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
cell_type_level1
Neutrophil             12418
Dendritic cell          2724
Fibroblast              1478
T cell                  1473
Natural killer cell     1115
B cell                   788
Name: count, dtype: int64


## scVI

In [9]:
adata.obsm['X_pca'] = adata.obsm['scVI']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scVI_cell_type1",save='_cell_type_dim15_eta10-scVI.png')
sc.pl.umap(adata, color='sample',title="scVI_sample",save='_sample_dim15_eta10-scVI.png')

In [8]:
import scanpy as sc

# 筛选 IAISR 细胞
adata_iaisr = adata[
    adata.obs["dataset"] == "IAISR"
].copy()

print(adata_iaisr)
print(adata_iaisr.obs["cell_type_level1"].value_counts())

# 使用 scVI embedding
adata_iaisr.obsm["X_pca"] = adata_iaisr.obsm["scVI"]

# 重新计算 neighbors 和 UMAP
sc.pp.neighbors(
    adata_iaisr,
    use_rep="scVI"
)
sc.tl.umap(adata_iaisr)
sc.pl.umap(
    adata_iaisr,
    color="sample",
    title="IAISR - scVI by sample",
    save="_IAISR_sample_scVI.png"
)
# 如果还有更细的 annotation，也可以一起看
sc.pl.umap(
    adata_iaisr,
    color="cell_type_level1",
    title="IAISR - scVI cell_type_level1",
    save="_IAISR_cell_type_level1_scVI.png"
)

AnnData object with n_obs × n_vars = 19996 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'conditions_combined'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'Harmony', 'LIGER', 'PCA', 'scANVI', 'scPoli', 'scVI'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
cell_type_level1
Neutrophil             12418
Dendritic cell          2724
Fibroblast              1478
T cell                  1473
Natural killer cell     1115
B cell                   788
Name: count, dtype: int64


## LIGER

In [10]:
adata.obsm['X_pca'] = adata.obsm['LIGER']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="LIGER_cell_type1",save='_cell_type_dim15_eta10-LIGER.png')
sc.pl.umap(adata, color='sample',title="LIGER_sample",save='_sample_dim15_eta10-LIGER.png')

In [9]:
import scanpy as sc

# 筛选 IAISR 细胞
adata_iaisr = adata[
    adata.obs["dataset"] == "IAISR"
].copy()

print(adata_iaisr)
print(adata_iaisr.obs["cell_type_level1"].value_counts())

# 使用 LIGER embedding
adata_iaisr.obsm["X_pca"] = adata_iaisr.obsm["LIGER"]

# 重新计算 neighbors 和 UMAP
sc.pp.neighbors(
    adata_iaisr,
    use_rep="LIGER"
)
sc.tl.umap(adata_iaisr)
sc.pl.umap(
    adata_iaisr,
    color="sample",
    title="IAISR - LIGER by sample",
    save="_IAISR_sample_LIGER.png"
)
# 如果还有更细的 annotation，也可以一起看
sc.pl.umap(
    adata_iaisr,
    color="cell_type_level1",
    title="IAISR - LIGER cell_type_level1",
    save="_IAISR_cell_type_level1_LIGER.png"
)

AnnData object with n_obs × n_vars = 19996 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'conditions_combined'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'Harmony', 'LIGER', 'PCA', 'scANVI', 'scPoli', 'scVI'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
cell_type_level1
Neutrophil             12418
Dendritic cell          2724
Fibroblast              1478
T cell                  1473
Natural killer cell     1115
B cell                   788
Name: count, dtype: int64


## scPoli

In [ ]:
# import pandas as pd
# import scanpy as sc

# csv_data = pd.read_csv(
#     "/home/lixiangyu/zr/Annotate/ground_truth_manual.csv",
#     header=0
# )

# # 清理 ground truth 里的细胞 ID
# csv_data.iloc[:, 0] = csv_data.iloc[:, 0].astype(str).str.replace(
#     r'(-\d+[_]\d+[_]\d+)+$',
#     '',
#     regex=True
# )

# # 统一命名
# csv_data.iloc[:, 1] = csv_data.iloc[:, 1].replace({
#     "SMC": "SMC(Fibromyocyte)",
#     "Fibromyocyte": "SMC(Fibromyocyte)",
#     "B_cell": "B_cell(Plasma_cell)",
#     "Plasma_cell": "B_cell(Plasma_cell)",
# })

# # 去重并设置 index
# csv_data = csv_data.iloc[:, :2].copy()
# csv_data.columns = ["cell_id", "ground_truth"]
# csv_data.drop_duplicates(subset="cell_id", keep="first", inplace=True)
# csv_data = csv_data.set_index("cell_id")

# print("\nGround truth counts:")
# print(csv_data["ground_truth"].value_counts())

# # 同样清理 adata.obs_names，生成一个用于匹配的 clean_cell_id
# adata.obs["clean_cell_id"] = adata.obs_names.astype(str).str.replace(
#     r'(-\d+[_]\d+[_]\d+)+$',
#     '',
#     regex=True
# )

# # 把 ground truth 映射到 adata.obs
# adata.obs["ground_truth"] = adata.obs["clean_cell_id"].map(csv_data["ground_truth"])

# print("\nMatched ground truth counts in adata:")
# print(adata.obs["ground_truth"].value_counts(dropna=False))

# # 用 scPoli embedding 画 UMAP
# adata.obsm["X_pca"] = adata.obsm["scPoli"]
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)

# # 画 ground truth，不用 cell_type_level1
# sc.pl.umap(
#     adata,
#     color="ground_truth",
#     title="scPoli_ground_truth",
#     save="_ground_truth_dim10_eta5_scPoli.png"
# )

# sc.pl.umap(
#     adata,
#     color="sample",
#     title="scPoli_sample",
#     save="_sample_dim10_eta5_scPoli.png"
# )

# # IAISR 子集
# adata_IAISR = adata[adata.obs["dataset"] == "IAISR"].copy()

# sc.pl.umap(
#     adata_IAISR,
#     color="ground_truth",
#     title="scPoli_IAISR_ground_truth",
#     save="_total_IAISR_ground_truth_dim10_eta5_scPoli.png"
# )

# sc.pl.umap(
#     adata_IAISR,
#     color="sample",
#     title="scPoli_IAISR_sample",
#     save="_total_IAISR_sample_dim10_eta5_scPoli.png"
# )


Ground truth counts:
ground_truth
T_cell                 83305
Monocyte               21864
SMC(Fibromyocyte)      21688
B_cell(Plasma_cell)    21360
Endothelial_cell       19305
Neutrophil             18708
Macrophage             18026
NK_cell                 8398
DC                      7449
Fibroblast              5757
Mast_cell               2044
NKT_cell                1437
Name: count, dtype: int64

Matched ground truth counts in adata:
ground_truth
NaN                    259900
T_cell                  86112
Monocyte                22898
SMC(Fibromyocyte)       21618
B_cell(Plasma_cell)     21589
Neutrophil              19600
Endothelial_cell        19524
Macrophage              18511
NK_cell                  9095
DC                       7431
Fibroblast               5254
Mast_cell                2137
NKT_cell                 1475
Name: count, dtype: int64


In [11]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim15_eta10-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim15_eta10-scPoli.png')

# adata_IAISR = adata[adata.obs['dataset'] == 'IAISR'].copy()

# sc.pl.umap(
#     adata_IAISR,
#     color='cell_type_level1',
#     title="scPoli_IAISR_cell_type1",
#     save='_total_IAISR_cell_type_dim10_eta5_scPoli.png'
# )

# sc.pl.umap(
#     adata_IAISR,
#     color='sample',
#     title="scPoli_IAISR_sample",
#     save='_total_IAISR_sample_dim10_eta5_scPoli.png')

In [ ]:
# import scanpy as sc

# # 筛选 IAISR 细胞
# adata_iaisr = adata[
#     adata.obs["dataset"] == "IAISR"
# ].copy()

# print(adata_iaisr)
# print(adata_iaisr.obs["cell_type_level1"].value_counts())

# # 使用 scPoli embedding
# adata_iaisr.obsm["X_pca"] = adata_iaisr.obsm["scPoli"]

# # 重新计算 neighbors 和 UMAP
# sc.pp.neighbors(
#     adata_iaisr,
#     use_rep="scPoli"
# )
# sc.tl.umap(adata_iaisr)
# sc.pl.umap(
#     adata_iaisr,
#     color="sample",
#     title="IAISR - scPoli by sample",
#     save="_IAISR_sample_scPoli.png"
# )
# # 如果还有更细的 annotation，也可以一起看
# sc.pl.umap(
#     adata_iaisr,
#     color="cell_type_level1",
#     title="IAISR - scPoli cell_type_level1",
#     save="_IAISR_cell_type_level1_scPoli.png"
# )

AnnData object with n_obs × n_vars = 19996 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'conditions_combined'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'Harmony', 'LIGER', 'PCA', 'scANVI', 'scPoli', 'scVI'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
cell_type_level1
Neutrophil             12418
Dendritic cell          2724
Fibroblast              1478
T cell                  1473
Natural killer cell     1115
B cell                   788
Name: count, dtype: int64


## PCA

In [12]:
adata.obsm['X_pca'] = adata.obsm['PCA']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="PCA_cell_type1",save='_cell_type_dim15_eta10-PCA.png')
sc.pl.umap(adata, color='sample',title="PCA_sample",save='_sample_dim15_eta10-PCA.png')

# UMAP-dim=10,eta=6

In [4]:
adata = sc.read_h5ad('./output_260420/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta6.h5ad')

## scPoli

In [5]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta6-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta6-scPoli.png')

# UMAP-dim=20,eta=5

In [2]:
adata = sc.read_h5ad('./output_260420/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim20-eta5.h5ad')

## scPoli

In [3]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim20_eta5-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim20_eta5-scPoli.png')

# UMAP-eta=8

In [40]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-eta8.h5ad')

## scPoli

In [45]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_eta8-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_eta8-scPoli.png')

# UMAP-dim=10,eta=4

In [ ]:
adata = sc.read_h5ad('/home/lixiangyu/zr/Annotate/ANNOTATE_new/3_small_integration/output_260420/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta4-noIAISR.h5ad')

## scPoli

In [ ]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta4_noIAISR-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta4_noIAISR-scPoli.png')

# UMAP-dim10

In [3]:
# adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10.h5ad')
adata = sc.read_h5ad('./output_260420/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10.h5ad')

## scPoli

In [ ]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1")


In [ ]:
sc.pl.umap(adata, color='sample',title="scPoli_sample")

# UMAP-dim=10,eta=6

In [54]:
adata = sc.read_h5ad('./output_1226/small-concat-ensembl-aggr-sparse-method-noCxG-mse-dim10-eta6.h5ad')

## scPoli

In [56]:
adata.obsm['X_pca'] = adata.obsm['scPoli']
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='cell_type_level1',title="scPoli_cell_type1",save='_cell_type_dim10_eta6-scPoli.png')
sc.pl.umap(adata, color='sample',title="scPoli_sample",save='_sample_dim10_eta6-scPoli.png')